# Stockout-Aware Forecasting for Perishable Goods: Reproduction Notebook

Reproduces all results of the paper in Google Colab. Run the cells in order.

All analysis code is in `src/`; this notebook only calls those scripts. Random seed: 42 (`src/config.py`).
Total run time on a free Colab CPU runtime: about 80 minutes (the forecasting step takes about 65).

## 1. Set up the project folder

**Option A – Google Drive** (data and results persist between sessions): the folder is found automatically.

**Option B – GitHub clone** (temporary session): comment out Option A and uncomment Option B.

In [1]:
import glob, os

# Option A: Google Drive. The project folder is found automatically anywhere in
# My Drive (the folder that contains src/config.py).
from google.colab import drive
drive.mount('/content/drive')
found = glob.glob('/content/drive/MyDrive/**/perishable-forecast/src/config.py', recursive=True)
newest = lambda f: max(os.path.getmtime(p) for p in glob.glob(os.path.join(os.path.dirname(f), '*.py')))
candidates = sorted(found, key=newest, reverse=True)   # most recently updated copy first
PROJECT_DIR = os.path.dirname(os.path.dirname(candidates[0])) if candidates else None

# Option B: clone from GitHub (comment out Option A above)
# !git clone https://github.com/Dr-Archana-Sasi/perishable-forecast.git /content/perishable-forecast
# PROJECT_DIR = '/content/perishable-forecast'

if PROJECT_DIR is None:
    raise FileNotFoundError('Project folder not found in My Drive; set PROJECT_DIR manually.')
if len(candidates) > 1:
    print('Several copies found; using the most recently updated one. Others:',
          [os.path.dirname(os.path.dirname(c)) for c in candidates[1:]])
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

Mounted at /content/drive
Several copies found; using the most recently updated one. Others: ['/content/drive/MyDrive/PostDocNewTEST/perishable-forecast']
Working directory: /content/drive/MyDrive/PostDocNewFinal/perishable-forecast


## 2. Install dependencies
Run at the start of every session.

In [2]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.8/501.8 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 4.5 MB/s eta 0:00:00


## 3. Download the dataset (FreshRetailNet-50K, CC BY 4.0)
Needed only once when using Google Drive.

In [3]:
!python src/download_data.py

README.md: 100% 4.87k/4.87k [00:00<00:00, 6.91MB/s]

data/train.parquet: downloading bytes:  65% 69.3M/106M [00:01<00:00, 106MB/s, 5.10MB/s  ] 
data/train.parquet: downloading bytes: 100% 78.5M/78.5M [00:01<00:00, 63.8MB/s, 7.52MB/s  ]
data/train.parquet: reconstructing file: 100% 106M/106M [00:01<00:00, 86.5MB/s, 10.3MB/s  ]

data/eval.parquet: downloading bytes:   5% 433k/8.44M [00:00<00:05, 1.34MB/s]
data/eval.parquet: downloading bytes: 100% 6.83M/6.83M [00:00<00:00, 17.6MB/s,  677kB/s  ]
data/eval.parquet: reconstructing file: 100% 8.44M/8.44M [00:00<00:00, 21.7MB/s,  837kB/s  ]
Generating train split: 100% 4500000/4500000 [00:18<00:00, 243487.04 examples/s]
Generating eval split: 100% 350000/350000 [00:00<00:00, 389969.64 examples/s]
Creating parquet from Arrow format: 100% 24/24 [00:21<00:00,  1.10ba/s]

[train] saved -> /content/drive/MyDrive/PostDocNewFinal/perishable-forecast/data/raw/train.parquet
  rows          : 4,500,000
  series        : 50,000
  date range    : 2024-03

## 4. Exploratory data analysis
Tables T1–T4, Figures F1–F5.

In [4]:
!python src/eda.py

Loaded 4,500,000 rows

                        Statistic      Value
        Rows (store-product-days)    4500000
             Store-product series      50000
                           Stores        898
                  Products (SKUs)        865
                           Cities         18
           First-level categories         32
          Second-level categories         84
           Third-level categories        233
                       Start date 2024-03-28
                         End date 2024-06-25
                             Days         90
              Zero-sales days (%)       4.46
      Open hours out of stock (%)      19.88
       Days with any stockout (%)      44.27
Days fully out of stock 06-22 (%)       4.04
     Days with discount < 1.0 (%)      51.53
                 Holiday days (%)      34.44
      Activity/promotion days (%)      37.84

stockout_hours    days  mean_rel_sales  median_rel_sales
             0 2507994          1.0139            0.9375
       

## 5. Demand recovery
Recovered daily demand, validation by synthetic censoring. Tables T5–T7, Figures F6–F7.

In [5]:
!python src/recover_demand.py

Loaded 4,850,000 rows: 50,000 series x 97 days

Validation on held-out stockout-free days (synthetic censoring):
                          method  validation_days   WAPE    bias
    Observed sales (no recovery)           491908 0.3858 -0.3858
                  Global profile           491908 0.2187 -0.0102
                Category profile           491908 0.2141 -0.0094
                  Series profile           491908 0.2103 -0.0094
Shrunk series profile (proposed)           491908 0.2099 -0.0099
Share of validation days recovered by projection: 0.835
Check: median relative gap between hourly sum and daily sales = 2.17e-08

Recovery on the real data:
split  store_product_days  censored_days_pct  recovered_by_projection_pct  recovered_by_fallback_pct  demand_uplift_all_days_pct  demand_uplift_censored_days_pct
train             4500000             44.267                       37.229                      7.038                      22.582                           49.511
 eval           

## 6. Forecasting
Seasonal naive, Empirical-28, ETS and LightGBM, each trained on recorded sales and on recovered demand. Tables T8–T10b, Figure F8. About 65 minutes.

In [6]:
!python src/forecast.py

Panel: 50,000 series x 97 days (train 90, evaluation 7)
  Seasonal naive  on sales     done in    0.2 s
  Empirical-28    on sales     done in    0.3 s
  ETS             on sales     done in  447.6 s
  LightGBM        on sales     done in 3210.5 s (772 rounds, 2,800,000 training rows)
  Seasonal naive  on recovered done in    0.2 s
  Empirical-28    on recovered done in    0.3 s
  ETS             on recovered done in  359.7 s
  LightGBM        on recovered done in 1840.7 s (437 rounds, 2,800,000 training rows)

Forecast accuracy (evaluation week):
                               eval_set          model       trained_on   rows   WAPE    bias  scaled_pinball  coverage_80
        A: all days vs recovered demand Seasonal naive   Recorded sales 350000 0.4071 -0.1583             NaN          NaN
B: stockout-free days vs recorded sales Seasonal naive   Recorded sales 206562 0.3891 -0.0077             NaN          NaN
        A: all days vs recovered demand   Empirical-28   Recorded sales 35000

## 7. Inventory simulation
Fill rate, waste and cost for each forecast set. Tables T11–T14, Figures F9–F10.

In [7]:
!python src/inventory_sim.py

8 forecast sets, 50,000 series x 7 days

Base case (unit cost 0.5, critical ratio 0.5, shelf life 1 day):
                       truth          model       trained_on  fill_rate  waste_pct_of_orders  stockout_day_rate  cost_per_unit_demand
            Recovered demand Seasonal naive Recovered demand     0.8036               0.2022             0.4746                0.2000
            Recovered demand Seasonal naive   Recorded sales     0.7173               0.1478             0.5690                0.2036
            Recovered demand   Empirical-28 Recovered demand     0.7351               0.1166             0.5670                0.1809
            Recovered demand   Empirical-28   Recorded sales     0.6674               0.0901             0.6490                0.1993
            Recovered demand            ETS Recovered demand     0.8257               0.1576             0.4775                0.1644
            Recovered demand            ETS   Recorded sales     0.7314               0.10

## 8. Record software versions
Writes `results/environment.txt`.

In [8]:
!python src/record_environment.py

python 3.13.16
platform Linux-6.6.122+-x86_64-with-glibc2.39
numpy==2.1.3
pandas==2.2.3
pyarrow==23.0.1
matplotlib==3.10.0
datasets==4.8.5
scikit-learn==1.6.1
lightgbm==4.6.0
statsforecast==2.1.1
scipy==1.16.3
statsmodels==0.15.0

Saved -> /content/drive/MyDrive/PostDocNewFinal/perishable-forecast/results/environment.txt


## 9. View the figures

In [ ]:
from IPython.display import display, Image
import glob
for f in sorted(glob.glob('results/figures/*.png'), key=lambda p: int(p.split('/F')[-1].split('_')[0])):
    print(f)
    display(Image(f, width=800))